# GoEmotions Long-Tail — 트랜스포머 + Log-Weighted Cross-Entropy

논문의 **아키텍처 독립성(architecture-independence)** 검증 실험. 동일한 가중 CE 계열
(CE / WCE / PWCE / SQCE / **LWCE** / **PLWCE** / CB / Focal)을 long-tailed **텍스트**
벤치마크에서 **from-scratch 소형 트랜스포머 인코더**로 비교한다 — CIFAR ResNet-32 실험의
백본 교체 버전.

- **데이터**: GoEmotions (`simplified`), single-label 필터 → 28-class 단일 라벨 분류
  (27 감정 + neutral), 자연 long-tail 불균형.
- **백본**: `transformer_text.build_text_transformer` (사전학습 없음).
- **손실**: `custom_losses.get_clf_loss` (`image_classification`와 동일 팩토리).
- **프로토콜**: 자연 불균형 단일 설정 (합성 IR 루프 없음), 5 seeds, F1-Macro 주지표.


In [ ]:
# === Cell 0: 환경 설정 ===
!pip install datasets optuna pandas openpyxl -q

import os, sys, json, re, math
from collections import Counter
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import matplotlib
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset
from sklearn.metrics import f1_score

import optuna
from optuna.samplers import GridSampler

from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/drive/MyDrive/imbalanced-data-LWCE'
# custom_losses.py / experiment_utils.py 는 REPO 루트, transformer_text.py 는 text_classification/.
sys.path.insert(0, f'{REPO}/text_classification')
sys.path.insert(0, REPO)

from custom_losses import get_clf_loss
from experiment_utils import (GradLogger, extended_metrics, OPTUNA_GRIDS,
                              grid_n_trials, LOSS_CONFIGS_11, ABLATION_ROWS)
from transformer_text import build_text_transformer

# --- 상수 설정 ---
DATASET      = 'go_emotions'
NUM_CLASSES  = 28          # 27 감정 + neutral (single-label 필터)
MAX_LEN      = 64
MIN_FREQ     = 2           # vocab 최소 토큰 빈도
BATCH_SIZE   = 64
NUM_WORKERS  = 0           # Colab notebook: 0 고정 (worker cleanup 오류 방지)
SEED         = 42
FINAL_EPOCHS = 50
WARMUP_EPOCHS = 3
SEEDS        = [42, 43, 44, 45, 46]

RESULTS_BASE = f'{REPO}/text_classification/results/GoEmotions'
os.makedirs(RESULTS_BASE, exist_ok=True)

CKPT_OPTUNA        = f'{RESULTS_BASE}/optuna_checkpoint.json'
CKPT_OPTUNA_TRIALS = f'{RESULTS_BASE}/optuna_trials.json'
CKPT_RESULTS       = f'{RESULTS_BASE}/results_checkpoint.json'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✓ Device: {device} | '
      f'{torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed(SEED)
matplotlib.use('Agg')

print(f'✓ Losses ({len(LOSS_CONFIGS_11)}): {LOSS_CONFIGS_11}')
print(f'✓ 전체 실행: {len(LOSS_CONFIGS_11)} losses × {len(SEEDS)} seeds = '
      f'{len(LOSS_CONFIGS_11) * len(SEEDS)} runs')

In [2]:
# === Cell 1: GoEmotions 로드, single-label 필터, vocab, 인코딩 ===
from datasets import load_dataset

TOKEN_RE = re.compile(r"[a-z0-9']+")

def tokenize(text):
    return TOKEN_RE.findall(text.lower())

def build_vocab(texts, min_freq=2):
    # train 텍스트에서 word-level vocab 구축 (0=<pad>, 1=<unk>)
    counter = Counter()
    for t in texts:
        counter.update(tokenize(t))
    itos = ['<pad>', '<unk>']
    for tok, c in counter.most_common():
        if c >= min_freq:
            itos.append(tok)
    stoi = {w: i for i, w in enumerate(itos)}
    return stoi, itos

def encode(text, stoi, max_len=MAX_LEN):
    # 토큰 → id, MAX_LEN 절단 후 우측 패딩 (1=<unk>, 0=<pad>)
    ids = [stoi.get(tok, 1) for tok in tokenize(text)][:max_len]
    if len(ids) < max_len:
        ids += [0] * (max_len - len(ids))
    return ids

def load_goemotions():
    # 최신 datasets는 네임스페이스 포함 정식 ID 필요 (bare 'go_emotions'는 HfUriError)
    ds = load_dataset('google-research-datasets/go_emotions', 'simplified')
    emotion_names = ds['train'].features['labels'].feature.names   # 28개 클래스명

    def to_single(split):
        # single-label 필터: 라벨이 정확히 1개인 예시만 사용
        texts, labels = [], []
        for ex in split:
            if len(ex['labels']) == 1:
                texts.append(ex['text'])
                labels.append(ex['labels'][0])
        return texts, labels

    tr_texts, tr_labels = to_single(ds['train'])
    va_texts, va_labels = to_single(ds['validation'])
    te_texts, te_labels = to_single(ds['test'])

    stoi, itos = build_vocab(tr_texts, MIN_FREQ)

    def make_loader(texts, labels, shuffle):
        X = torch.tensor([encode(t, stoi) for t in texts], dtype=torch.long)
        y = torch.tensor(labels, dtype=torch.long)
        return DataLoader(TensorDataset(X, y), batch_size=BATCH_SIZE,
                          shuffle=shuffle, num_workers=NUM_WORKERS)

    train_loader = make_loader(tr_texts, tr_labels, True)
    val_loader   = make_loader(va_texts, va_labels, False)
    test_loader  = make_loader(te_texts, te_labels, False)

    class_counts = [0] * NUM_CLASSES
    for l in tr_labels:
        class_counts[l] += 1

    return train_loader, val_loader, test_loader, class_counts, stoi, itos, emotion_names

train_loader, val_loader, test_loader, class_counts, stoi, itos, EMOTION_NAMES = load_goemotions()
VOCAB_SIZE = len(itos)

_nz = [c for c in class_counts if c > 0]
print(f'✓ Vocab 크기: {VOCAB_SIZE}')
print(f'✓ Train/Val/Test (single-label): '
      f'{len(train_loader.dataset)}/{len(val_loader.dataset)}/{len(test_loader.dataset)}')
print(f'✓ train 샘플 >0 클래스: {len(_nz)}/{NUM_CLASSES}')
print(f'✓ 클래스 분포 (train): min={min(_nz)}, max={max(class_counts)}, '
      f'IR={max(class_counts)/max(1, min(_nz)):.1f}:1')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


✓ Vocab 크기: 12226
✓ Train/Val/Test (single-label): 36308/4548/4590
✓ train 샘플 >0 클래스: 28/28
✓ 클래스 분포 (train): min=39, max=12823, IR=328.8:1


In [3]:
# === Cell 2: 클래스 분포 시각화 ===
def visualize_class_distribution(class_counts, names, dataset_name='GoEmotions'):
    counts_arr = np.array(class_counts)
    nz = counts_arr[counts_arr > 0]

    print('=' * 60)
    print(f'{dataset_name} (single-label) — 클래스 분포')
    print('=' * 60)
    print(f'Min: {nz.min()} | Max: {counts_arr.max()} | '
          f'Ratio: {counts_arr.max() / nz.min():.1f}:1 | Total: {counts_arr.sum():,}')

    # Many/Medium/Few 그룹 (텍스트 도메인 기준)
    many_mask   = counts_arr >= 500
    medium_mask = (counts_arr >= 100) & (counts_arr < 500)
    few_mask    = counts_arr < 100
    print(f'  Many-shot   (n>=500):      {many_mask.sum():2d} classes')
    print(f'  Medium-shot (100<=n<500):  {medium_mask.sum():2d} classes')
    print(f'  Few-shot    (n<100):       {few_mask.sum():2d} classes')

    # 막대 그래프 (샘플 수 내림차순, log scale)
    order = np.argsort(counts_arr)[::-1]
    colors = ['green' if counts_arr[i] >= 500 else ('orange' if counts_arr[i] >= 100 else 'red')
              for i in order]
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.bar(range(len(order)), counts_arr[order], color=colors, alpha=0.75)
    ax.set_yscale('log')
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels([names[i] for i in order], rotation=60, ha='right', fontsize=7)
    ax.set_ylabel('# samples (log scale)')
    ax.set_title(f'{dataset_name} single-label long-tail distribution')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{RESULTS_BASE}/class_distribution.png', dpi=100, bbox_inches='tight')
    plt.close()
    print(f'\n✓ 분포 저장: {RESULTS_BASE}/class_distribution.png')

visualize_class_distribution(class_counts, EMOTION_NAMES)

GoEmotions (single-label) — 클래스 분포
Min: 39 | Max: 12823 | Ratio: 328.8:1 | Total: 36,308
  Many-shot   (n>=500):      19 classes
  Medium-shot (100<=n<500):   5 classes
  Few-shot    (n<100):        4 classes

✓ 분포 저장: /content/drive/MyDrive/imbalanced-data-LWCE/text_classification/results/GoEmotions/class_distribution.png


In [ ]:
# === Cell 3: 모델 및 평가 함수 정의 ===
# 3차 피드백 §4.3 지표: Macro-F1, Balanced Acc, G-Mean, Minority Recall, Worst-class Acc, Per-class Acc
#   → G-Mean/Worst-class/Per-class는 experiment_utils.extended_metrics 로 전 도메인 통일
def build_model():
    return build_text_transformer(VOCAB_SIZE, NUM_CLASSES, pad_idx=0, max_len=MAX_LEN)

def _predict(model, loader):
    model.eval()
    yt, yp = [], []
    with torch.no_grad():
        for xb, yb in loader:
            yp.extend(model(xb.to(device)).argmax(dim=1).cpu().tolist())
            yt.extend(yb.tolist())
    return np.array(yt), np.array(yp)

def compute_val_metrics(model, loader, num_classes, class_counts_train=None):
    y_true, y_pred = _predict(model, loader)
    result = {
        'Top1_Acc': float((y_true == y_pred).mean()),
        'F1_Macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
    }
    # Per_Class_Acc / G_Mean / Worst_Acc / Balanced_Acc
    result.update(extended_metrics(y_true, y_pred, num_classes))

    if class_counts_train is not None:
        pca = np.array(result['Per_Class_Acc'])
        sorted_idx = np.argsort(np.array(class_counts_train))[::-1]   # many→few
        n = len(sorted_idx)
        result['Many_Acc']   = float(pca[sorted_idx[:n // 3]].mean())
        result['Medium_Acc'] = float(pca[sorted_idx[n // 3:2 * n // 3]].mean())
        # Minority-class Recall = Few 그룹의 recall 평균
        result['Few_Acc']    = float(pca[sorted_idx[2 * n // 3:]].mean())
    return result

def compute_val_acc(model, loader):
    # Optuna 목적함수 / 모델 선택 기준 스칼라 = F1-Macro (balanced_acc 아님)
    y_true, y_pred = _predict(model, loader)
    return f1_score(y_true, y_pred, average='macro', zero_division=0)

print('✓ 모델 및 평가 함수 정의 완료')
print('  지표: Top1 / Balanced / F1-Macro / G-Mean / Worst-class / Many·Medium·Few(=Minority Recall)')
print('  Optuna·모델선택 기준: F1-Macro | 그룹: train count tertile split')

In [ ]:
# === Cell 4: train_model 정의 (AdamW + warmup-cosine + gradient 계측) ===
# 3차 피드백 §4.7: Weight Explosion 완화 "mechanism"을 증명하려면 성능 수치만으로는 부족.
#   → epoch별 Gradient Norm / Minority·Majority Gradient Ratio를 학습 중에 기록 (GradLogger).
#     둘 다 학습이 끝나면 복구 불가능.
#   이 노트북은 grad clipping을 쓰므로 clip_grad_norm_의 리턴값(클리핑 전 norm)을 그대로 넘긴다.

def train_model(loss_name, class_counts, train_loader, val_loader, num_classes,
                alpha=1.0, gamma=2.0, eps=0.1, tau=1.0,
                epochs=FINAL_EPOCHS, lr=5e-4, tag='', seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)

    model = build_model().to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)

    # linear warmup (WARMUP_EPOCHS) → cosine decay
    def lr_lambda(epoch):
        if epoch < WARMUP_EPOCHS:
            return (epoch + 1) / WARMUP_EPOCHS
        progress = (epoch - WARMUP_EPOCHS) / max(1, epochs - WARMUP_EPOCHS)
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    criterion = get_clf_loss(loss_name, class_counts, alpha=alpha, gamma=gamma,
                             eps=eps, tau=tau)
    glog = GradLogger(class_counts, num_classes, device)

    best_val_f1, best_state = 0.0, None
    # 주의: history 키 'val_balanced_acc'는 레거시 이름. 실제 값은 F1-Macro.
    history = {'epoch': [], 'train_loss': [], 'val_balanced_acc': [],
               'grad_norm': [], 'grad_many': [], 'grad_few': [], 'grad_ratio': []}
    pbar = tqdm(range(epochs),
                desc=f'{loss_name} (α={alpha:.2f}, γ={gamma:.2f}, ε={eps:.2f}, τ={tau:.2f})',
                leave=False)

    for epoch in pbar:
        model.train()
        glog.reset()
        train_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            logits.retain_grad()          # 샘플별 로짓 기울기 관측 (Proposition 4)
            loss = criterion(logits, yb)
            loss.backward()
            # clip_grad_norm_는 '클리핑 전' total norm을 리턴 → 재계산 없이 그대로 사용
            total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            glog.update(logits, yb, grad_norm=total_norm)
            optimizer.step()
            train_loss += loss.item() * yb.size(0)
        train_loss /= len(train_loader.dataset)

        val_f1 = compute_val_acc(model, val_loader)        # F1-Macro
        history['epoch'].append(epoch)
        history['train_loss'].append(train_loss)
        history['val_balanced_acc'].append(val_f1)
        for k, v in glog.epoch_end().items():
            history[k].append(v)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        scheduler.step(); pbar.update()

    if best_state:
        model.load_state_dict(best_state)
    model.eval()
    return model, history, best_val_f1

print('✓ train_model 정의 완료 (AdamW/warmup-cosine/grad-clip=1.0, 선택 기준: F1-Macro)')
print('  기록: train_loss, val F1-Macro, grad_norm, grad_many/few, grad_ratio(=Few/Many)')

In [ ]:
# === Cell 5: Optuna 탐색 (자연 불균형 단일 설정, 목적함수 = F1-Macro on val) ===
# 3차 피드백 §3.3(재현 가능한 selection procedure) / §4.6(sensitivity analysis) 대응:
#   · proxy = train의 stratified 부분집합, 선택 기준 = val split의 F1-Macro → test는 일절 미사용
#   · study.trials 전체를 저장 → α·ε·τ sensitivity 곡선을 추가 학습 없이 확보
os.environ['TQDM_DISABLE'] = '1'

PROXY_EPOCHS        = 10
PROXY_SUBSET_RATIO  = 0.20
PROXY_MIN_PER_CLASS = 5

# 탐색 그리드는 experiment_utils.OPTUNA_GRIDS로 전 도메인 통일
#   (1D 30 trials, combined만 2D 10x6=60). n_trials는 grid 크기와 정확히 일치해야 함.
GRIDS = OPTUNA_GRIDS

# 기존 체크포인트 로드 후 '빠진 study만' 추가 실행 (완료된 pwce/plwce/focal 재탐색 방지)
optuna_best, optuna_trials = {}, {}
if os.path.exists(CKPT_OPTUNA):
    with open(CKPT_OPTUNA) as f:
        optuna_best = json.load(f)
    print('Optuna 체크포인트 로드:', optuna_best)
if os.path.exists(CKPT_OPTUNA_TRIALS):
    with open(CKPT_OPTUNA_TRIALS) as f:
        optuna_trials = json.load(f)

pending = [k for k in GRIDS if k not in optuna_best]

if not pending:
    print('✓ 모든 study 완료 — 추가 탐색 없음')
else:
    print(f'  탐색 필요: {pending}')

    # stratified proxy: 클래스별 최소 PROXY_MIN_PER_CLASS개 보장 → tail 클래스 소멸 방지
    y_all = train_loader.dataset.tensors[1].numpy()
    rng = np.random.RandomState(SEED)
    proxy_pos = []
    for c in np.unique(y_all):
        pos_c = np.where(y_all == c)[0]
        n_take = max(int(round(PROXY_SUBSET_RATIO * len(pos_c))),
                     min(len(pos_c), PROXY_MIN_PER_CLASS))
        n_take = min(n_take, len(pos_c))
        proxy_pos.extend(rng.choice(pos_c, size=n_take, replace=False).tolist())
    rng.shuffle(proxy_pos)
    proxy_loader = DataLoader(Subset(train_loader.dataset, proxy_pos),
                              batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    print(f'  proxy: {len(proxy_pos)} samples ({PROXY_SUBSET_RATIO:.0%} of train), '
          f'{len(np.unique(y_all))} classes (min {PROXY_MIN_PER_CLASS}/class 보장)')

    def run_study(loss_name, grid):
        n_trials = grid_n_trials(grid)              # grid 크기와 정확히 일치
        def objective(trial):
            kw = {p: trial.suggest_float(p, min(g), max(g)) for p, g in grid.items()}
            m, _, _ = train_model(loss_name, class_counts, proxy_loader, val_loader,
                                  NUM_CLASSES, epochs=PROXY_EPOCHS, **kw)
            return compute_val_acc(m, val_loader)                  # F1-Macro
        study = optuna.create_study(direction='maximize',
                                    sampler=GridSampler(grid),     # MedianPruner 병행 금지
                                    study_name=f'goemotions_{loss_name}')
        study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
        return study

    for loss_name in pending:
        study = run_study(loss_name, GRIDS[loss_name])
        optuna_best[loss_name] = dict(study.best_params)
        # sensitivity 분석용 — 'if t.value is not None' (0.0 trial 필터링 버그 방지)
        optuna_trials[loss_name] = [{'params': t.params, 'value': t.value}
                                    for t in study.trials if t.value is not None]
        ps = ', '.join(f'{k}={v:.4f}' for k, v in study.best_params.items())
        print(f'  [{loss_name}] best {ps} (val F1-Macro = {study.best_value:.4f}, '
              f'{len(optuna_trials[loss_name])} trials 저장)')

    with open(CKPT_OPTUNA, 'w') as f:
        json.dump(optuna_best, f, indent=2)
    with open(CKPT_OPTUNA_TRIALS, 'w') as f:
        json.dump(optuna_trials, f, indent=2)
    print('  Optuna best:', optuna_best)
    print(f'  저장 → {CKPT_OPTUNA} , {CKPT_OPTUNA_TRIALS}')

os.environ['TQDM_DISABLE'] = '0'
print('✓ Optuna 탐색 완료 (ce/wce/sqce/lwce/cb는 파라미터 없음 → 탐색 불필요)')

In [ ]:
# === Cell 6: 전체 Loss × Seed 비교 실험 (자연 불균형) ===
# 3차 피드백 §4.2 baseline 요구: CE, WCE, Focal, CB, (LDAM 또는 Logit Adjustment), LWCE, PLWCE, ES-LWCE
#   + §4.5 ablation의 Combined(= log compression O / α O / ε O) 행
LOSS_CONFIGS = LOSS_CONFIGS_11

# ── 체크포인트 로드 ──────────────────────────────────
if os.path.exists(CKPT_RESULTS):
    with open(CKPT_RESULTS) as f:
        ckpt_data = json.load(f)
    print(f'결과 체크포인트 로드: {len(ckpt_data)}개 완료된 실행')
else:
    ckpt_data = {}
    print('체크포인트 없음 — 새로 시작')

# ── 완료된 결과 복원 ────────────────────────────────
all_results   = {l: {} for l in LOSS_CONFIGS}
all_histories = {l: {} for l in LOSS_CONFIGS}
for run_key, rd in ckpt_data.items():
    try:
        loss_str, seed_str = run_key.rsplit('_', 1)      # '{loss}_s{seed}'
        seed = int(seed_str[1:])
        if loss_str in all_results:
            all_results[loss_str][seed]   = rd['metrics']
            all_histories[loss_str][seed] = rd['history']
    except Exception:
        pass

total_runs = len(LOSS_CONFIGS) * len(SEEDS)
print(f'\nFull experiment: {len(LOSS_CONFIGS)} losses × {len(SEEDS)} seeds = {total_runs} runs')
print(f'완료: {len(ckpt_data)}/{total_runs}')
print('=' * 60)

# ── 실험 실행 ────────────────────────────────────────
for loss_name in LOSS_CONFIGS:
    # optuna_best[loss_name]은 해당 손실의 파라미터만 담고 있음 → 없으면 기본값
    # optuna_best[loss_name]은 해당 손실의 파라미터만 담고 있음 → 없으면 기본값
    best  = optuna_best.get(loss_name, {})
    alpha = best.get('alpha', 1.0)
    gamma = best.get('gamma', 2.0)
    eps   = best.get('eps',   0.1)
    tau   = best.get('tau',   1.0)

    for seed in SEEDS:
        run_key = f'{loss_name}_s{seed}'
        if run_key in ckpt_data:
            continue

        print(f'\n[{loss_name}] seed={seed} '
              f'(α={alpha:.2f}, γ={gamma:.2f}, ε={eps:.2f}, τ={tau:.2f})...')
        model, history, _ = train_model(
            loss_name, class_counts, train_loader, val_loader, NUM_CLASSES,
            alpha=alpha, gamma=gamma, eps=eps, tau=tau, epochs=FINAL_EPOCHS,
            tag=f'{loss_name}_s{seed}', seed=seed)

        metrics = compute_val_metrics(model, test_loader, NUM_CLASSES,
                                      class_counts_train=class_counts)
        metrics.update({'alpha': alpha, 'gamma': gamma, 'eps': eps, 'tau': tau})
        # §4.7 Class Weight Distribution 분석용 — 실제 사용된 정규화 가중치 기록
        w = get_clf_loss(loss_name, class_counts, alpha=alpha, gamma=gamma,
                         eps=eps, tau=tau).get_weights()
        metrics['class_weights'] = None if w is None else w.tolist()

        all_results[loss_name][seed]   = metrics
        all_histories[loss_name][seed] = history
        ckpt_data[run_key] = {'metrics': metrics, 'history': history}
        with open(CKPT_RESULTS, 'w') as f:
            json.dump(ckpt_data, f)

        print(f'  F1={metrics["F1_Macro"]:.4f} | Worst={metrics["Worst_Acc"]:.4f} | '
              f'G-Mean={metrics["G_Mean"]:.4f} | Few={metrics.get("Few_Acc", 0):.4f} | '
              f'grad_ratio={history["grad_ratio"][-1]:.2f} → 저장됨')

print('\n✓ 전체 훈련 완료')

In [ ]:
# === Cell 7: 결과 집계, 시각화 및 저장 ===
METRIC_KEYS = ['Top1_Acc', 'Balanced_Acc', 'F1_Macro', 'G_Mean', 'Worst_Acc',
               'Many_Acc', 'Medium_Acc', 'Few_Acc']

def get_group_indices(class_counts):
    # train count 기준 상위/중위/하위 1/3 인덱스 반환
    sorted_idx = np.argsort(np.array(class_counts))[::-1]
    n = len(sorted_idx)
    return sorted_idx[:n // 3], sorted_idx[n // 3:2 * n // 3], sorted_idx[2 * n // 3:]

many_idx, medium_idx, few_idx = get_group_indices(class_counts)
agg_results = {}
colors_loss = plt.cm.tab20(np.linspace(0, 1, len(LOSS_CONFIGS)))

print('최종 결과 요약 (mean ± std, 그룹 = train count 상위/중위/하위 1/3 tertile split)')
print('=' * 112)
print(f'{"Loss":10s} | {"Top1":^15} | {"F1-Macro":^15} | {"Worst":^6} | {"G-Mean":^6} | '
      f'{"Balanc":^6} | {"Many":^6} | {"Med":^6} | {"Few":^6} | n')
print('-' * 112)

for loss_name in LOSS_CONFIGS:
    seed_metrics = [all_results[loss_name][s] for s in SEEDS if s in all_results[loss_name]]
    if not seed_metrics:
        print(f'{loss_name:10s} | (미완료)')
        continue
    # Per_Class_Acc에서 그룹별/파생 지표 재계산 (구버전 체크포인트 호환)
    for m in seed_metrics:
        pca = np.array(m.get('Per_Class_Acc', []))
        if len(pca) > 0:
            m['Many_Acc']   = float(pca[many_idx].mean())
            m['Medium_Acc'] = float(pca[medium_idx].mean())
            m['Few_Acc']    = float(pca[few_idx].mean())
            m.setdefault('G_Mean',    float(np.exp(np.mean(np.log(pca + 1e-12)))))
            m.setdefault('Worst_Acc', float(pca.min()))
    agg = {}
    for key in METRIC_KEYS:
        vals = [m.get(key, 0.0) for m in seed_metrics]
        agg[f'{key}_mean'] = float(np.mean(vals))
        agg[f'{key}_std']  = float(np.std(vals))
    agg_results[loss_name] = agg
    print(f'{loss_name:10s} | '
          f'{agg["Top1_Acc_mean"]:.4f}±{agg["Top1_Acc_std"]:.4f} | '
          f'{agg["F1_Macro_mean"]:.4f}±{agg["F1_Macro_std"]:.4f} | '
          f'{agg["Worst_Acc_mean"]:.4f} | {agg["G_Mean_mean"]:.4f} | '
          f'{agg["Balanced_Acc_mean"]:.4f} | {agg["Many_Acc_mean"]:.4f} | '
          f'{agg["Medium_Acc_mean"]:.4f} | {agg["Few_Acc_mean"]:.4f} | {len(seed_metrics)}')

# ── JSON 저장 ──────────────────────────────────────
with open(f'{RESULTS_BASE}/results_agg.json', 'w') as f:
    json.dump(agg_results, f, indent=2)
per_seed_data = {l: {str(s): all_results[l].get(s, {}) for s in SEEDS} for l in LOSS_CONFIGS}
with open(f'{RESULTS_BASE}/results_per_seed.json', 'w') as f:
    json.dump(per_seed_data, f, indent=2)

# ── Excel 저장 ────────────────────────────────────
summary_rows, per_seed_rows = [], []
for loss_name in LOSS_CONFIGS:
    if loss_name not in agg_results:
        continue
    agg = agg_results[loss_name]
    row = {'Loss': loss_name}
    for key in METRIC_KEYS:
        row[f'{key}_Mean'] = f"{agg[f'{key}_mean']:.4f}"
        row[f'{key}_Std']  = f"{agg[f'{key}_std']:.4f}"
    summary_rows.append(row)
    for seed in SEEDS:
        m = all_results[loss_name].get(seed, {})
        if m:
            r = {'Loss': loss_name, 'Seed': seed}
            r.update({k: f"{m.get(k, 0):.4f}" for k in METRIC_KEYS})
            per_seed_rows.append(r)
with pd.ExcelWriter(f'{RESULTS_BASE}/results.xlsx', engine='openpyxl') as writer:
    pd.DataFrame(summary_rows).to_excel(writer, sheet_name='Summary_Agg', index=False)
    pd.DataFrame(per_seed_rows).to_excel(writer, sheet_name='Per_Seed', index=False)

# ── 학습 곡선 (mean ± std shading) ─────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
for l_idx, loss_name in enumerate(LOSS_CONFIGS):
    seed_hists = [all_histories[loss_name][s] for s in SEEDS if s in all_histories[loss_name]]
    if not seed_hists:
        continue
    arr = np.array([h['val_balanced_acc'] for h in seed_hists])   # 레거시 키, 값은 F1-Macro
    mean, std = arr.mean(0), arr.std(0)
    ep = np.arange(len(mean))
    ax.plot(ep, mean, label=loss_name, color=colors_loss[l_idx], alpha=0.85)
    ax.fill_between(ep, mean - std, mean + std, color=colors_loss[l_idx], alpha=0.15)
ax.set_xlabel('Epoch'); ax.set_ylabel('Val F1-Macro')
ax.set_title(f'GoEmotions-LT Training Curves (mean±std, n≤{len(SEEDS)} seeds)')
ax.legend(fontsize=7, ncol=2); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_BASE}/training_curves.png', dpi=150, bbox_inches='tight')
plt.close()

# ── F1-Macro / Worst-class 비교 (3차 피드백이 지목한 두 핵심 지표) ──
losses = [l for l in LOSS_CONFIGS if l in agg_results]
x = np.arange(len(losses))
bar_colors = [colors_loss[LOSS_CONFIGS.index(l)] for l in losses]
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, key, title in zip(axes, ['F1_Macro', 'Worst_Acc'],
                          ['F1-Macro (key metric 1)', 'Worst-class Acc (key metric 2)']):
    means = [agg_results[l][f'{key}_mean'] for l in losses]
    stds  = [agg_results[l][f'{key}_std']  for l in losses]
    ax.bar(x, means, yerr=stds, capsize=4, alpha=0.78, color=bar_colors)
    ce_mean = agg_results.get('ce', {}).get(f'{key}_mean')
    if ce_mean is not None:
        ax.axhline(ce_mean, color='gray', linestyle='--', linewidth=1, alpha=0.7,
                   label='CE baseline')
        ax.legend(fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(losses, rotation=40, ha='right')
    ax.set_ylabel(key); ax.set_title(title); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_BASE}/f1_macro_comparison.png', dpi=150, bbox_inches='tight')
plt.close()

# ── Many/Medium/Few 그룹별 정확도 비교 ──────────────────
fig, ax = plt.subplots(figsize=(11, 5))
width = 0.25
group_colors = {'Many': '#2ecc71', 'Medium': '#f39c12', 'Few': '#e74c3c'}
for g_idx, (group, key) in enumerate([('Many', 'Many_Acc_mean'),
                                       ('Medium', 'Medium_Acc_mean'),
                                       ('Few', 'Few_Acc_mean')]):
    vals = [agg_results[l].get(key, 0.0) for l in losses]
    ax.bar(x + (g_idx - 1) * width, vals, width, label=group,
           color=group_colors[group], alpha=0.78)
ax.set_xticks(x); ax.set_xticklabels(losses, rotation=40, ha='right')
ax.set_ylabel('Accuracy'); ax.set_title('GoEmotions-LT Many/Medium/Few Accuracy (tertile split)')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_BASE}/group_accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.close()

print('\n✓ 결과 저장 완료')
print(f'  JSON:  {RESULTS_BASE}/results_agg.json , results_per_seed.json')
print(f'  Excel: {RESULTS_BASE}/results.xlsx  (Summary_Agg / Per_Seed 시트)')
print(f'  PNG:   class_distribution / training_curves / f1_macro_comparison / group_accuracy_comparison')
print(f'  Checkpoint: {CKPT_RESULTS}')

In [ ]:
# === Cell 8: 3차 피드백 §4.5~4.8 분석 (mechanism / sensitivity / ablation / 통계) ===
from itertools import cycle
from scipy import stats

# ── (1) §4.7 Optimization Stability — Gradient Norm & Minority/Majority Ratio ──
# "단순 성능 향상만으로는 Weight Explosion 완화를 증명 못 한다"는 지적에 대한 직접 증거.
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for l_idx, loss_name in enumerate(LOSS_CONFIGS):
    hs = [all_histories[loss_name][s] for s in SEEDS if s in all_histories[loss_name]]
    if not hs or 'grad_norm' not in hs[0]:
        continue
    c = colors_loss[l_idx]
    for ax, key, ylab in zip(axes,
                             ['train_loss', 'grad_norm', 'grad_ratio'],
                             ['Train Loss', 'Gradient Norm (pre-clip)',
                              'Gradient Ratio (Few / Many)']):
        arr = np.array([h[key] for h in hs], dtype=float)
        mean, std = np.nanmean(arr, 0), np.nanstd(arr, 0)
        ep = np.arange(len(mean))
        ax.plot(ep, mean, label=loss_name, color=c, alpha=0.85)
        ax.fill_between(ep, mean - std, mean + std, color=c, alpha=0.12)
        ax.set_xlabel('Epoch'); ax.set_ylabel(ylab); ax.grid(alpha=0.3)
axes[1].set_yscale('log')     # WCE의 explosion을 보이려면 로그 스케일이 필요
axes[2].axhline(1.0, color='k', linestyle=':', linewidth=1, alpha=0.6)
axes[0].set_title('Training Loss'); axes[1].set_title('Gradient Norm (log scale)')
axes[2].set_title('Minority/Majority Gradient Ratio')
axes[0].legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig(f'{RESULTS_BASE}/optimization_stability.png', dpi=150, bbox_inches='tight')
plt.close()

print('■ Gradient 통계 (마지막 epoch, seed 평균)')
print(f'{"Loss":10s} | {"grad_norm":>10} | {"Few/Many ratio":>14}')
print('-' * 42)
grad_summary = {}
for loss_name in LOSS_CONFIGS:
    hs = [all_histories[loss_name][s] for s in SEEDS if s in all_histories[loss_name]]
    if not hs or 'grad_norm' not in hs[0]:
        continue
    gn = float(np.mean([h['grad_norm'][-1] for h in hs]))
    gr = float(np.nanmean([h['grad_ratio'][-1] for h in hs]))
    grad_summary[loss_name] = {'grad_norm_final': gn, 'grad_ratio_final': gr}
    print(f'{loss_name:10s} | {gn:10.4f} | {gr:14.3f}')

# ── (2) §4.7 Class Weight Distribution ──
fig, ax = plt.subplots(figsize=(11, 5))
order = np.argsort(np.array(class_counts))[::-1]      # many → few
markers = cycle(['o', 's', '^', 'v', 'D', 'P', '*', 'X', '<', '>', 'h'])
for l_idx, loss_name in enumerate(LOSS_CONFIGS):
    m = next((all_results[loss_name][s] for s in SEEDS if s in all_results[loss_name]), None)
    if not m:
        continue
    w = m.get('class_weights')
    w = np.ones(NUM_CLASSES) if w is None else np.array(w)   # None = 균일 가중치(CE/focal/logitadj)
    ax.plot(range(NUM_CLASSES), w[order], marker=next(markers), markersize=3,
            label=loss_name, color=colors_loss[l_idx], alpha=0.85, linewidth=1.2)
ax.set_yscale('log')
ax.set_xlabel('Class (sorted: many → few)')
ax.set_ylabel('Normalized weight $\\tilde{w}_c$ (log scale)')
ax.set_title('Class Weight Distribution — mean-normalized $\\tilde{w}_c = C w_c / \\sum_j w_j$')
ax.legend(fontsize=7, ncol=2); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_BASE}/class_weight_distribution.png', dpi=150, bbox_inches='tight')
plt.close()

print('\n■ 가중치 동적 범위 (rare/frequent 비율) — Proposition 3')
for loss_name in LOSS_CONFIGS:
    m = next((all_results[loss_name][s] for s in SEEDS if s in all_results[loss_name]), None)
    if not m or m.get('class_weights') is None:
        continue
    w = np.array(m['class_weights'])
    print(f'  {loss_name:10s} w_rare/w_freq = {w[order[-1]] / w[order[0]]:10.2f}')

# ── (3) §4.6 Sensitivity Analysis (Optuna trial 전체 재활용, 추가 학습 없음) ──
if os.path.exists(CKPT_OPTUNA_TRIALS):
    with open(CKPT_OPTUNA_TRIALS) as f:
        trials_data = json.load(f)
    oneD = [(l, p) for l, ps in [('pwce', 'alpha'), ('plwce', 'alpha'),
                                 ('eslwce', 'eps'), ('focal', 'gamma'), ('logitadj', 'tau')]
            for p in [ps] if l in trials_data]
    if oneD:
        fig, axes = plt.subplots(1, len(oneD), figsize=(4.2 * len(oneD), 3.8))
        axes = np.atleast_1d(axes)
        for ax, (loss_name, param) in zip(axes, oneD):
            ts = sorted(trials_data[loss_name], key=lambda t: t['params'][param])
            xs = [t['params'][param] for t in ts]
            ys = [t['value'] for t in ts]
            ax.plot(xs, ys, marker='o', markersize=3, color='#2d4860')
            best = max(ts, key=lambda t: t['value'])
            ax.axvline(best['params'][param], color='#e74c3c', linestyle='--', linewidth=1,
                       label=f"best={best['params'][param]:.3f}")
            if param == 'eps':
                ax.set_xscale('log')
            ax.set_xlabel(param); ax.set_ylabel('proxy val F1-Macro')
            ax.set_title(f'{loss_name}: {param} sensitivity', fontsize=10)
            ax.legend(fontsize=8); ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(f'{RESULTS_BASE}/sensitivity_1d.png', dpi=150, bbox_inches='tight')
        plt.close()
        print('\n✓ sensitivity_1d.png 저장 (proxy 기준 — 논문 그림엔 full-training sweep 권장)')

    # combined: 2D (alpha × eps) 히트맵
    if 'combined' in trials_data:
        ts = trials_data['combined']
        A = sorted({t['params']['alpha'] for t in ts})
        E = sorted({t['params']['eps'] for t in ts})
        Z = np.full((len(E), len(A)), np.nan)
        for t in ts:
            Z[E.index(t['params']['eps']), A.index(t['params']['alpha'])] = t['value']
        fig, ax = plt.subplots(figsize=(6.5, 4.5))
        im = ax.imshow(Z, aspect='auto', origin='lower', cmap='viridis')
        ax.set_xticks(range(len(A))); ax.set_xticklabels([f'{a:.2f}' for a in A])
        ax.set_yticks(range(len(E))); ax.set_yticklabels([f'{e:.2f}' for e in E])
        ax.set_xlabel('alpha (PLWCE)'); ax.set_ylabel('eps (ES-LWCE)')
        ax.set_title('Combined: proxy val F1-Macro')
        plt.colorbar(im, ax=ax)
        plt.tight_layout()
        plt.savefig(f'{RESULTS_BASE}/sensitivity_combined_2d.png', dpi=150, bbox_inches='tight')
        plt.close()
        print('✓ sensitivity_combined_2d.png 저장')

# ── (4) §4.5 Ablation 표 (Log Compression / Power α / Smoothing ε) ──
ABLATION = [('wce', 'X', 'X', 'X'), ('lwce', 'O', 'X', 'X'), ('plwce', 'O', 'O', 'X'),
            ('eslwce', 'O', 'X', 'O'), ('combined', 'O', 'O', 'O')]
print('\n■ Ablation (3차 피드백 §4.5)')
print(f'{"Method":10s} | {"LogComp":^7} | {"Power α":^7} | {"Smooth ε":^8} | '
      f'{"F1-Macro":^15} | {"Worst":^6}')
print('-' * 72)
ablation_rows = []
for name, lc, pw, sm in ABLATION:
    if name not in agg_results:
        continue
    a = agg_results[name]
    print(f'{name:10s} | {lc:^7} | {pw:^7} | {sm:^8} | '
          f'{a["F1_Macro_mean"]:.4f}±{a["F1_Macro_std"]:.4f} | {a["Worst_Acc_mean"]:.4f}')
    ablation_rows.append({'Method': name, 'LogCompression': lc, 'Power_alpha': pw,
                          'Smoothing_eps': sm,
                          'F1_Macro': f'{a["F1_Macro_mean"]:.4f}±{a["F1_Macro_std"]:.4f}',
                          'Worst_Acc': f'{a["Worst_Acc_mean"]:.4f}'})

# ── (5) §4.8 통계 검정 (Friedman + Wilcoxon, seed를 블록으로) ──
# 주의: 정석은 '여러 데이터셋'을 블록으로 두는 것. 단일 데이터셋에서는 seed 블록이라
#       검정력이 낮음 → CIFAR/network 등 전 도메인 결과가 모이면 그때 데이터셋 블록으로 재실행할 것.
present = [l for l in LOSS_CONFIGS if len(all_results[l]) == len(SEEDS)]
if len(present) >= 3:
    M = np.array([[all_results[l][s]['F1_Macro'] for s in SEEDS] for l in present])  # (losses, seeds)
    try:
        chi2, pval = stats.friedmanchisquare(*[M[i, :] for i in range(len(present))])
        print(f'\n■ Friedman test (F1-Macro, {len(present)} losses × {len(SEEDS)} seeds)')
        print(f'  chi2={chi2:.3f}, p={pval:.4g}  → '
              f'{"손실 간 유의미한 차이 있음" if pval < 0.05 else "유의미한 차이 없음"}')
    except Exception as e:
        print('\n[Friedman skipped]', e)
    ranks = np.array([stats.rankdata(-M[:, j]) for j in range(len(SEEDS))]).mean(0)
    print('\n■ Mean Rank (1위가 최고) / vs PLWCE Wilcoxon')
    ref = 'plwce'
    for i, l in enumerate(sorted(range(len(present)), key=lambda i: ranks[i])):
        name = present[l]
        line = f'  {int(round(ranks[l])):2d}. {name:10s} rank={ranks[l]:.2f}'
        if ref in present and name != ref:
            try:
                _, p = stats.wilcoxon(M[l, :], M[present.index(ref), :])
                line += f'  (p={p:.3f})'
            except Exception:
                pass
        print(line)

# ── 저장 ──
with open(f'{RESULTS_BASE}/analysis_summary.json', 'w') as f:
    json.dump({'grad_summary': grad_summary, 'ablation': ablation_rows}, f, indent=2)
print(f'\n✓ 분석 저장 완료 → {RESULTS_BASE}/analysis_summary.json')
print('  PNG: optimization_stability / class_weight_distribution / '
      'sensitivity_1d / sensitivity_combined_2d')